# Installation

For this notebook, you'll need to install several Python packages. I assume you already have a package manager like miniconda. If not, here is a [link](https://www.anaconda.com/docs/getting-started/miniconda/install/overview) to select your operating system and to follow the directions there (for **Windows** users: make sure to click the option to add miniconda to your environment paths during the installation process!).

I will now show you how to create a new conda environment, but you can also install the packages in your environment of choice, if you already have one.

Open a terminal window or powershell and copy and paste these commands in:

> `conda create -n equi python=3.14` 
>
> `conda activate equi`

Now we'll install PyTorch. For users with an NVIDIA GPU, follow the guide below, otherwise skip to the next section of this text.

Make sure you have CUDA 13.0 or newer installed. You can check this by opening your terminal or powershell and typing `nvidia-smi` + hitting Enter. It should be written in the top right corner. If you don't have it installed, make sure to [download](https://developer.nvidia.com/cuda-downloads) (or find a newer version) and install it.

> For Linux users:
>
> `pip3 install torch torchvision`
>
> For Windows users:
>
> `pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu130`

For users with an AMD GPU, make sure you have ROCm 7.2 or newer, however PyTorch for the GPU is only available on Unix systems here. If you are on Windows, skip to the next section of text.

> `pip3 install torch torchvision --index-url https://download.pytorch.org/whl/rocm7.2`

Finally, you can run PyTorch on the CPU only:

> Linux users:
>
> `pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cpu`
>
> MacOS and Windows users:
>
> `pip3 install torch torchvision`

Finally, we can install the other packages via the terminal / powershell:

> `pip install torch_geometric`
>
> `pip install e3nn`
>
> `pip install rdkit`
>
> `pip install matplotlib`
>
> `pip install pandas`

If you are running this notebook in Google Colab, running the cell below should install everything you need.

In [ ]:
import sys
if "google.colab" in sys.modules:
    %pip install rdkit
    %pip install torch_geometric
    %pip install e3nn

    !git clone https://github.com/SGeorgiev00/PhD_Tutorials.git

In [ ]:
from pathlib import Path
import math
import os
import shutil
from typing import Any

from rdkit import Chem
from rdkit.Chem import Atom, Bond

import numpy as np
import pandas as pd

import torch
from torch import Tensor, LongTensor
from torch import nn
from torch import optim
from torch.nn import functional as F

from torch_geometric import nn as geometric_nn
from torch_geometric import data
from torch_geometric import loader
from torch_geometric import datasets

from e3nn import o3
from e3nn import nn as e3nn_nn
from e3nn.math import soft_one_hot_linspace

import matplotlib as mpl
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt

torch.manual_seed(123)
torch.cuda.manual_seed_all(123)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# clean execution
if os.path.exists("data/train_md17/"):
    shutil.rmtree("data/train_md17/")

if os.path.exists("data/test_md17/"):
    shutil.rmtree("data/test_md17/")

# Introduction to Graph Neural Networks and Geometric Deep Learning

**Important note:** 

Please keep in mind that this notebook has the goal of only showcasing some of the tools we can use to build GNNs and SE(3)-equivariant GNNs. We'll be using toy datasets and some randomly picked hyperparameters for our models. Therefore, any direct comparison of the results is erroneous.

## 1. Demonstration of GNNs and PyTorch Geometric

In this section, we'll recreate a popular directed message passing neural network from scratch. Afterwards, we'll use one of the models provided by [PyTorch Geometric](https://pytorch-geometric.readthedocs.io/en/latest/).

Let's first start with creating a dataset. We have some medicinal chemistry data about inhibitors of the BRK protein (an interesting kinase target for breast cancer). The provided data points are the SMILES strings of the tested compounds and a binary classification of whether they bound to the target or not.

We'll turn the SMILES strings into graph data points using the **rdkit** and **torch_geometric** packages. We will recreate the features and a rudimentary version of the D-MPNN from [ChemProp](https://pubs.acs.org/doi/10.1021/acs.jcim.3c01250).

In [ ]:
if "google.colab" in sys.modules:
    additional_path = "PhD_Tutorials/04_GNNs_GDL"
else:
    additional_path = ""
    
brk = pd.read_csv(os.path.join(additional_path, "data/brk.csv"))
print(f"{brk.activity.mean() * 100:.2f} % actives")
brk.head()

Now to create our own dataset. We'll be following the [documentation](https://pytorch-geometric.readthedocs.io/en/latest/notes/create_dataset.html) for this. We'll featurize our graphs with the atom and bond features from ChemProp.

We'll inherit from `torch_geometric.data.InMemoryDataset` (you can also inherit from `torch_geometric.data.Dataset` for large datasets and save every data point as its own file). We need to define the four methods specified explicitly in the docs:

* `raw_file_names()` - a list of files in the dataset's `raw_dir` (raw) that need to be found in order to skip the download of the raw data
* `processed_file_names()` - a list of files in the dataset's `processed_dir` (processed) that need to be found in order to skip the processing of the raw data
* `download()` - downloads the raw data into `raw_dir` (raw)
* `process()` - processes the raw data and saves it into the `processed_dir` (processed)

There are also options to implement `transform`, `pre_transform` and `pre_filter` callables, should a user want to modify their data.

Each graph needs to be saved to its own `torch_geometric.data.Data` object. Let's see how to do this in practice. In the code below, the file names will be able to be parsed as lists of strings in order to show you how you can process multiple files together.

In [ ]:
def process_to_chemprop_graph(smiles_list: np.ndarray, y_data: np.ndarray) -> list[data.Data]:
    """Creates the node, edge and directed edge features of a graph from ChemProp from an rdkit Mol object and lists of the mostly one-hot-encoded attributes as defined in ChemProp

    Args
    ----
    smiles_list : np.ndarray
        A list of the SMILES strings to process to graphs
    y_data : np.ndarray
        A list of the binary activity data

    Returns
    -------
    list[torch_geometric.data.Data]
        A list of the processed graphs as Data objects
    """
    
    data_list = []
    
    mols = [Chem.MolFromSmiles(smi) for smi in smiles_list]

    # load some predefined dictionaries (a lot a one-hot encoded and it's easier to reference a pre-saved dictionary)
    dict_atom_chiral_types_to_int = {v: k for k, v in Chem.ChiralType.values.items()}
    dict_atom_hybridization_types_to_int = {v: k for k, v in Chem.HybridizationType.values.items()}
    dict_bond_stereo_types_to_int = {v: k for k, v in Chem.BondStereo.values.items()}

    df_atom_types = pd.read_csv("utils/atom_one_hot_encodings_atom_type.csv")
    df_degrees = pd.read_csv("utils/atom_one_hot_encodings_degree.csv")
    df_formal_charges = pd.read_csv("utils/atom_one_hot_encodings_formal_charge.csv")
    df_chiral_tags = pd.read_csv("utils/atom_one_hot_encodings_chiral_tag.csv")
    df_num_hydrogens = pd.read_csv("utils/atom_one_hot_encodings_num_hydrogens.csv")
    df_hybridizations = pd.read_csv("utils/atom_one_hot_encodings_hybridization.csv")
    df_bond_types = pd.read_csv("utils/bond_one_hot_encodings_bond_type.csv")
    df_bond_stereos = pd.read_csv("utils/bond_one_hot_encodings_bond_stereo.csv")

    for i, mol in enumerate(mols):
        # node features
        node_features = []

        for atom in mol.GetAtoms():
            node_data = []

            if Atom.GetSymbol(atom) in df_atom_types.columns:
                node_data.extend(df_atom_types[Atom.GetSymbol(atom)].to_list())
            else:
                node_data.extend(df_atom_types["unknown"].to_list())

            if Atom.GetTotalDegree(atom) in df_degrees.columns:
                node_data.extend(df_degrees[Atom.GetTotalDegree(atom)].to_list())
            else:
                node_data.extend(df_degrees["unknown"].to_list())

            if Atom.GetFormalCharge(atom) in df_formal_charges.columns:
                node_data.extend(df_formal_charges[Atom.GetFormalCharge(atom)].to_list())
            else:
                node_data.extend(df_formal_charges["unknown"].to_list())

            if dict_atom_chiral_types_to_int[Atom.GetChiralTag(atom)] in df_chiral_tags.columns:
                node_data.extend(df_chiral_tags[dict_atom_chiral_types_to_int[Atom.GetChiralTag(atom)]].to_list())
            else:
                node_data.extend(df_chiral_tags["unknown"].to_list())

            if Atom.GetTotalNumHs(atom) in df_num_hydrogens.columns:
                node_data.extend(df_num_hydrogens[Atom.GetTotalNumHs(atom)].to_list())
            else:
                node_data.extend(df_num_hydrogens["unknown"].to_list())

            if dict_atom_hybridization_types_to_int[Atom.GetHybridization(atom)] in df_hybridizations.columns:
                node_data.extend(df_hybridizations[dict_atom_hybridization_types_to_int[Atom.GetHybridization(atom)]].to_list())
            else:
                node_data.extend(df_hybridizations["unknown"])

            node_data.append(int(Atom.GetIsAromatic(atom)))
            node_data.append(Atom.GetMass(atom) / 100)

            node_features.append(node_data)

        node_features = torch.tensor(node_features, dtype=torch.float)

        # edge indices and features
        edge_indices = []
        edge_attrs = []

        total_num_bond_features = 1 + len(df_bond_types.columns) + 2 + len(df_bond_stereos.columns)

        for bond in mol.GetBonds():  # my implementation uses .GetBonds() directly, as we are only working with single molecule datasets. For reaction, solvation and chelation datasets (i.e. multi-component datasets), this will not work and will require a work-around.
            starting_idx = Bond.GetBeginAtomIdx(bond)
            ending_idx = Bond.GetEndAtomIdx(bond)

            # append twice because edge_index is directional (has source and target)
            edge_indices.append([starting_idx, ending_idx])
            edge_indices.append([ending_idx, starting_idx])

            if bond is None:  # safeguard as in their implementation - I guess for reaction, solvation and chelation datasets (i.e. multi-component datasets), but I don't see the point of it for single molecule datasets.
                bond_data = [0 for _ in range(total_num_bond_features)]
                bond_data[0] = 1
                edge_attrs.append(bond_data)
                continue

            bond_data = [0]

            bond_data.extend(df_bond_types[str(Bond.GetBondTypeAsDouble(bond))].to_list())

            bond_data.append(int(Bond.GetIsConjugated(bond)))
            bond_data.append(int(Bond.IsInRing(bond)))
            
            if dict_bond_stereo_types_to_int[Bond.GetStereo(bond)] in df_bond_stereos.columns:
                bond_data.extend(df_bond_stereos[dict_bond_stereo_types_to_int[Bond.GetStereo(bond)]].to_list())
            else:
                bond_data.extend(df_bond_stereos["unknown"].to_list())

            edge_attrs.append(bond_data)
            edge_attrs.append(bond_data)

        edge_index = torch.tensor(edge_indices)
        edge_index = edge_index.t().to(torch.long).view(2, -1)  # will have the shape [2, however many edges * 2] - the first index is the source node, the second index is the target node
        edge_attr = torch.tensor(edge_attrs, dtype=torch.float)

        y = torch.tensor(y_data[[i]], dtype=torch.float).view(-1, 1)

        data_list.append(
            data.Data(
                x=node_features,
                edge_index=edge_index,
                edge_attr=edge_attr,
                y=y
            )
        )
    
    return data_list

In [ ]:
class ChemPropDataset(data.InMemoryDataset):
    def __init__(self, root, raw_files: list[str], processed_files: list[str], original_files: list[str],
                 transform=None, pre_transform=None, pre_filter=None) -> None:
        if not os.path.exists(root):
            os.makedirs(root)

        self.raw_files = raw_files
        self.processed_files = processed_files
        self.original_files = original_files

        super().__init__(root, transform, pre_transform, pre_filter)
        self.load(self.processed_paths[0])

    @property
    def raw_file_names(self) -> None:
        return self.raw_files
    
    @property
    def processed_file_names(self) -> None:
        return self.processed_files
    
    def download(self) -> None:
        for i in range(len(self.raw_files)):
            shutil.copy(os.path.abspath(self.original_files[i]), os.path.join(self.raw_dir, self.raw_files[i]))

    def process(self) -> None:
        raw_data = self.raw_paths
        raw_data = pd.concat([pd.read_csv(csv_file) for csv_file in raw_data], axis=0)

        data_list = process_to_chemprop_graph(raw_data.smiles.to_numpy(), raw_data.activity.to_numpy())
        self.save(data_list, self.processed_paths[0])

In [ ]:
brk_dataset = ChemPropDataset(
    root=os.path.join(additional_path, "data/brk_chemprop"),
    raw_files=["brk.csv"],
    processed_files=["brk_processed.pt"],
    original_files=[os.path.join(additional_path, "data/brk.csv")]
)
brk_dataset

Let's look at an example:

In [ ]:
brk_dataset[0]

Now let's split the data randomly into a training, validation and test set in a ratio of 8:1:1.

**Note:** This is way of non-stratification of the target labels and random splitting is generally not a good idea and only serves our demonstration purposes.

In [ ]:
brk_dataset = brk_dataset.shuffle()

brk_train_dataset = brk_dataset[:int(0.8*len(brk_dataset))]
brk_valid_dataset = brk_dataset[int(0.8*len(brk_dataset)):int(0.9*len(brk_dataset))]
brk_test_dataset = brk_dataset[int(0.9*len(brk_dataset)):]

brk_train_loader = loader.DataLoader(brk_train_dataset, batch_size=64, shuffle=True)
brk_valid_loader = loader.DataLoader(brk_valid_dataset, batch_size=64)  # shuffle is off by default
brk_test_loader = loader.DataLoader(brk_test_dataset, batch_size=64)

Let's implement some classes and functions that will help us have a more minimalistic code later on - we'll define some metrics we'll want to use in this notebook, as well as some functions for training and testing our models.

In [ ]:
class Metrics:
    def __init__(self, metrics_to_use: list[str]) -> None:
        self.stored = {metric: [] for metric in metrics_to_use}

        self.calculate = {
            "f1": self.f1,
            "mae": self.mae,
            "rmse": self.rmse,
            "cosine_similarity": self.cosine_sim
        }

    def __repr__(self) -> str:
        return f"Metrics(stored={self.stored})"

    def f1(self, pred: Tensor, true: Tensor, batch_size: int = 1) -> None:
        out = torch.round(torch.sigmoid(pred)).to("cpu").detach().flatten()
        true = true.flatten().cpu()

        tp = (true * out).sum()
        # tn = ((1 - true) * (1 - out)).sum()
        fp = ((1 - true) * out).sum()
        fn = (true * (1 - out)).sum()

        epsilon = 1e-7

        precision = tp / (tp + fp + epsilon)
        recall = tp / (tp + fn + epsilon)

        f1 = 2* (precision*recall) / (precision + recall + epsilon)
        
        self.stored["f1"].append(f1)
    
    def mae(self, pred: Tensor, true: Tensor, batch_size: int = 1) -> None:
        mae = torch.mean(torch.abs(pred.to("cpu") - true.cpu()))

        self.stored["mae"].append(mae)
    
    def rmse(self, pred: Tensor, true: Tensor, batch_size: int = 1) -> None:
        rmse = torch.sqrt(torch.mean((pred.to("cpu") - true.cpu())**2))
        
        self.stored["rmse"].append(rmse)
    
    def cosine_sim(self, pred: Tensor, true: Tensor, batch_size: int = 1) -> None:
        cos_sim = torch.mean(F.cosine_similarity(pred.cpu(), true.cpu())).item()
        
        self.stored["cosine_similarity"].append(cos_sim)
    
    # or add any other metric you'd like here

In [ ]:
def train_model(model: nn.Module, criterion: nn.Module, criterion_args: list, optimizer: optim.Optimizer, optimizer_args: list,
          train_loader: loader.DataLoader, valid_loader: loader.DataLoader, metrics_list: list[str],
          target: str, criterion_is_cos_sim: bool = False,
          epochs: int = 100) -> tuple[nn.Module, list[float]] | nn.Module:
    model = model
    criterion = criterion(*criterion_args)
    optimizer = optimizer(model.parameters(), *optimizer_args)

    train_loss_tracker = []
    valid_loss_tracker = []

    for epoch in range(1, epochs+1):
        metrics = Metrics(metrics_list)

        train_loss_list = []
        model.train()
        
        for batch in train_loader:
            optimizer.zero_grad()

            batch.to(DEVICE)

            pred = model(batch)

            true = batch[target]

            if criterion_is_cos_sim:
                ones = torch.ones(len(pred), device=pred.device)
                loss = criterion(pred, true, ones)
            else:
                true = true.view(-1, 1)
                loss = criterion(pred, true)
            
            train_loss_list.append(loss.item())
            loss.backward()
            optimizer.step()

        valid_loss_list = []
        model.eval()
        with torch.no_grad():
            for batch in valid_loader:
                batch.to(DEVICE)

                pred = model(batch)
                true = batch[target]

                if criterion_is_cos_sim:
                    ones = torch.ones(len(pred), device=pred.device)
                    loss = criterion(pred, true, ones)
                else:
                    true = true.view(-1, 1)
                    loss = criterion(pred, true)
                
                valid_loss_list.append(loss.item())

                for metric in metrics_list:
                    metrics.calculate[metric](pred, true, len(batch))

        train_loss = sum(train_loss_list) / len(train_loader)
        train_loss_tracker.append(train_loss)

        valid_loss = sum(valid_loss_list) / len(valid_loader)
        valid_loss_tracker.append(valid_loss)

        if epoch % 10 == 0:
            print(f"Epoch {epoch:3d}: Train loss: {train_loss:7.4f} | Validation loss: {valid_loss:7.4f} | ", end="")
            for metric in metrics_list:
                print(f"Validation {metric}: {np.mean(metrics.stored[metric]):7.4f} | ", end="")
            print()

    return model, train_loss_tracker, valid_loss_tracker

In [ ]:
def test_model(model: nn.Module, test_loader: loader.DataLoader, metrics_list: list[Any], target: str) -> object:
    model = model
    model.eval()

    metrics = Metrics(metrics_list)

    total_pred = []
    total_true = []

    with torch.no_grad():
        for batch in test_loader:
            batch.to(DEVICE)

            pred = model(batch)
            true = batch[target]

            total_pred.append(pred.cpu())
            total_true.append(true.cpu())

    total_pred = torch.cat(total_pred, dim=0)
    total_true = torch.cat(total_true, dim=0)

    for metric in metrics_list:
        metrics.calculate[metric](total_pred, total_true)

    return metrics

We can now use a pre-defined convolution model from `torch_geometric` or build our own.

For this quick demo, we'll use the implemented Graph Convolution Network ([Kipf and Welling](https://arxiv.org/abs/1609.02907)) that only accepts node features in order to build our GNN. We refer to the [GNN Cheatsheet](https://pytorch-geometric.readthedocs.io/en/latest/cheatsheet/gnn_cheatsheet.html) for a list of the available models in the library.

In [ ]:
class GCN(nn.Module):
    def __init__(
            self,
            gnn_in_features: int,
            gnn_hid_features: int,
            gnn_num_layers: int,
            mlp_hid_layers: list[int],
            out_features: int,
            mlp_dropout: float
    ) -> None:
        super().__init__()

        gnn_blocks = [
            (geometric_nn.conv.GCNConv(gnn_in_features, gnn_hid_features, add_self_loops=True), "x, edge_index -> x")
        ]

        for i in range(gnn_num_layers-1):
            gnn_blocks.append(
                (geometric_nn.conv.GCNConv(gnn_hid_features, gnn_hid_features, add_self_loops=True), "x, edge_index -> x")
                                )
            gnn_blocks.append(nn.LeakyReLU(inplace=True))

        gnn_blocks.append(
            (geometric_nn.global_add_pool, "x, batch -> x")
                         )
        
        mlp_hid_layers += [out_features]
        
        gnn_blocks.append(
            (nn.Linear(gnn_hid_features, mlp_hid_layers[0]))
                          )

        mlp_blocks = []
        for (in_feat, out_feat) in zip(mlp_hid_layers[:-1], mlp_hid_layers[1:]):
            mlp_blocks.append(nn.BatchNorm1d(in_feat))
            mlp_blocks.append(nn.ReLU())
            mlp_blocks.append(nn.Dropout(mlp_dropout))
            mlp_blocks.append(nn.Linear(in_feat, out_feat))

        self.gnn_model = geometric_nn.Sequential("x, edge_index, batch", gnn_blocks)
        self.mlp_model = nn.Sequential(*mlp_blocks)

    def forward(self, batch: data.Batch) -> Tensor:
        x = self.gnn_model(batch.x, batch.edge_index, batch.batch)
        return self.mlp_model(x)

In [ ]:
simple_model = GCN(
    gnn_in_features=brk_dataset[0].x.shape[1],
    gnn_hid_features= 100,
    gnn_num_layers=3,
    mlp_hid_layers=[50, 20, 10],
    out_features=1,
    mlp_dropout=0.2
).to(DEVICE)

In [ ]:
trained_simple_model, train_loss, valid_loss = train_model(
    simple_model, 
    nn.BCEWithLogitsLoss,
    [(brk_train_dataset.y == 0).sum() / (brk_train_dataset.y == 1).sum()], 
    optim.Adam, 
    [0.001], 
    brk_train_loader, 
    brk_valid_loader,
    ["f1"],
    "y"
)

In [ ]:
epochs = [i+1 for i in range(len(train_loss))]

plt.plot(epochs, train_loss, color="blue", label="train loss")
plt.plot(epochs, valid_loss, color="red", label="valid loss")

plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("GCN model, BRK")

plt.legend()
plt.show()

In [ ]:
metrics = test_model(trained_simple_model, brk_test_loader, ["f1"], "y")
metrics

Here, we can showcase a very slow re-implementation of ChemProp from scratch.

## 2. Demonstration of a Naïve GNN, an Invariant GNN and an Equivariant GNN

In [ ]:
train_md17_dataset = datasets.MD17(root=Path(_dh[-1]) / os.path.join(additional_path, "data/train_md17/"),
                           name="aspirin CCSD",
                           train=True,
                           force_reload=True)
print(len(train_md17_dataset))
train_md17_dataset[0]

In [ ]:
test_md17_dataset = datasets.MD17(root=Path(_dh[-1]) / os.path.join(additional_path, "data/test_md17/"),
                          name="aspirin CCSD",
                          train=False,
                          force_reload=True)
print(len(test_md17_dataset))

Let's present at some data points visually.

In [ ]:
# modified from Volkamerlab

def to_perceived_brightness(rgb: np.ndarray) -> np.ndarray:
    """
    Auxiliary function, useful for choosing label colors
    with good visibility
    """
    r, g, b = rgb
    return 0.1 * r + 0.8 * g + 0.1


def plot_point_cloud_and_vectors_3d(
    fig: mpl.figure.Figure,
    ax_pos: int,
    color: np.ndarray,
    pos: np.ndarray,
    vectors: np.ndarray,
    cmap: str = "plasma",
    point_size: float = 180.0,
    label_axes: bool = False,
    annotate_points: bool = True,
    remove_axes_ticks: bool = True,
    cbar_label: str = "",
) -> mpl.axis.Axis:
    """Visualize colored 3D point clouds.

    Parameters
    ----------
    fig : mpl.figure.Figure
        The figure for which a new axis object is added for plotting
    ax_pos : int
        Three-digit integer specifying axis layout and position
        (see docs for `mpl.figure.Figure.add_subplot`)
    color : np.ndarray
        The point colors as a float array of shape `(N,)`
    pos : np.ndarray
        The point xyz-coordinates as an array
    vectors : np.ndarray
        The vectors from each xyz coordinate as an array
    cmap : str, optional
        String identifier for a matplotlib colormap.
        Is used to map the values in `color` to rgb colors.
        , by default "plasma"
    point_size : float, optional
        The size of plotted points, by default 180.0
    label_axes : bool, optional
        whether to label x,y and z axes by default False
    annotate_points : bool, optional
        whether to label points with their index, by default True
    cbar_label : str, optional
        label for the colorbar, by default ""

    Returns
    -------
    mpl.axis.Axis
        The new axis object for the 3D point cloud plot.
    """
    cmap = plt.get_cmap(cmap)
    ax = fig.add_subplot(ax_pos, projection="3d")
    x, y, z = pos
    u, v, w = pos + vectors

    sc = ax.scatter(x, y, z, c=color, cmap=cmap, s=point_size)
    ax.quiver(x, y, z, u, v, w, length=1.5, normalize=True, color="green", alpha=0.7)

    if remove_axes_ticks:
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_zticks([])
    if label_axes:
        ax.set_xlabel("$x$")
        ax.set_ylabel("$y$")
        ax.set_zlabel("$z$")

    plt.colorbar(sc, location="bottom", shrink=0.6, anchor=(0.5, 2), label=cbar_label)
    if annotate_points:
        _colors = sc.cmap(color)
        rgb = _colors[:, :3].transpose()
        brightness = to_perceived_brightness(rgb)
        for i, (xi, yi, zi, li) in enumerate(zip(x, y, z, brightness)):
            ax.text(xi, yi, zi, str(i), None, color=[1 - li] * 3, ha="center", va="center")
    return ax


# testing
fig = plt.figure(figsize=(10, 8))

for i, ax_pos in enumerate([221, 222, 223, 224]):
    pos = train_md17_dataset[i].pos.T
    vectors = train_md17_dataset[i].force.T  
    color = train_md17_dataset[i].z
    plot_point_cloud_and_vectors_3d(fig, ax_pos, color, pos, vectors)

fig.suptitle("First four data points")

fig.tight_layout()
plt.show()

In [ ]:
# stolen from Volkamerlab

# use rotations along z-axis as demo e(3) transformation
def rotation_matrix_z(theta: float) -> Tensor:
    """Generates a rotation matrix and returns
    a corresponing tensor. The rotation is about the $z$-axis.
    (https://en.wikipedia.org/wiki/Rotation_matrix)

    Parameters
    ----------
    theta : float
        the angle of rotation in radians.

    Returns
    -------
    Tensor
        the rotation matrix as float tensor.
    """
    return torch.tensor(
        [
            [math.cos(theta), -math.sin(theta), 0],
            [math.sin(theta), math.cos(theta), 0],
            [0, 0, 1],
        ]
    )


Let's rotate this molecule around the z-axis.

In [ ]:
# Some data point
sample_data = train_md17_dataset[800].clone()

# apply an E(3) transformation
rotated_sample_data = sample_data.clone()

deg = 90
theta = deg * (math.pi / 180)
rotated_sample_data.pos = rotated_sample_data.pos @ rotation_matrix_z(theta)
rotated_sample_data.force = rotated_sample_data.force @ rotation_matrix_z(theta)

# make a plot that demonstrates rotation
fig = plt.figure(figsize=(10, 8))

ax1 = plot_point_cloud_and_vectors_3d(fig, 121, sample_data.z, sample_data.pos.T, sample_data.force.T)
ax1.set_title("Sample input $(X, Z)$")

ax2 = plot_point_cloud_and_vectors_3d(fig, 122, rotated_sample_data.z, rotated_sample_data.pos.T, rotated_sample_data.force.T)
ax2.set_title("Rotated input $(X, g(Z))$")

fig.tight_layout()
plt.show()

Now we'll want to preprocess the data, so that it is centered at the molecule's center of mass and so it can have the appropriate features for our models.

In [ ]:
MAP_Z_TO_ATOMIC_MASS = {
        1: 1.0080,
        6: 12.011,
        8: 15.999
    }

MASSES_ASPIRIN_ATOMS = torch.tensor([MAP_Z_TO_ATOMIC_MASS[z.item()] for z in train_md17_dataset[0].z], dtype=torch.float)
TOTAL_MASS_ASPIRIN = MASSES_ASPIRIN_ATOMS.sum()

def get_center_of_mass(pos: Tensor) -> Tensor:
    x, y, z = pos.T

    total_x_mass = torch.sum(x * MASSES_ASPIRIN_ATOMS)
    total_y_mass = torch.sum(y * MASSES_ASPIRIN_ATOMS)
    total_z_mass = torch.sum(z * MASSES_ASPIRIN_ATOMS)

    center_of_mass = torch.tensor([total_x_mass/TOTAL_MASS_ASPIRIN, 
                                   total_y_mass/TOTAL_MASS_ASPIRIN, 
                                   total_z_mass/TOTAL_MASS_ASPIRIN], dtype=torch.float).view((1, -1))

    return center_of_mass

In [ ]:
CUTOFF = 4.5


def get_closest_nodes(pos: Tensor, cutoff: float = CUTOFF) -> tuple[LongTensor, Tensor]:
    distances = torch.cdist(pos, pos, p=2)

    # set the diagonals to inf so they are never taken.
    replacements = torch.tensor([float("inf") for _ in range(len(distances))], dtype=torch.float)

    mask = torch.diag(torch.ones_like(replacements))
    distances = mask * torch.diag(replacements) + (1. - mask) * distances
    
    # get the indices of the nodes that are closer than the cutoff
    pairs = (distances < cutoff).nonzero().T
    
    # get the scalars (distances) for the scalar edge features
    source, target = pairs
    edge_attr_scalars = distances[source, target].view((-1, 1))

    return pairs, edge_attr_scalars


Moreover, we have to normalize the data and we need to get the mean and the standard deviation now using the functions we just defined. We'll be using 85% of the modelling set for training and the rest for validation here.

In [ ]:
class DataNormalizer:
    def __init__(self):
        self.mean = 0.0
        self.std = 0.0

    def fit(self, x: Tensor) -> None:
        self.mean = x.mean()
        self.std = x.std()

    def normalize(self, x: Tensor) -> Tensor:
        return (x - self.mean) / self.std

In [ ]:
new_pos = []

for i in range(len(train_md17_dataset[:int(len(train_md17_dataset)*0.85)])):
    new_pos.append((train_md17_dataset[i].pos - get_center_of_mass(train_md17_dataset[i].pos)).tolist())

new_pos = torch.tensor(new_pos, dtype=torch.float)

edge_attr = []
for i in range(len(train_md17_dataset[:int(len(train_md17_dataset)*0.85)])):
    _, edge_dist = get_closest_nodes(new_pos[i])
    edge_attr.append(edge_dist.flatten())

edge_attr = torch.cat(edge_attr)

In [ ]:
NORMALIZER_ENERGY = DataNormalizer()
NORMALIZER_EDGE_ATTR = DataNormalizer()

NORMALIZER_ENERGY.fit(train_md17_dataset.energy)
NORMALIZER_EDGE_ATTR.fit(edge_attr)

In [ ]:
def preprocess(data: data.Data) -> data.Data:
    data.pos = data.pos - get_center_of_mass(data.pos)
    data.edge_index, edge_attr = get_closest_nodes(data.pos)
    data.energy = NORMALIZER_ENERGY.normalize(data.energy)
    data.edge_attr = NORMALIZER_EDGE_ATTR.normalize(edge_attr)
    return data

In [ ]:
modelling_md17_dataset = datasets.MD17(root=Path(_dh[-1]) / os.path.join(additional_path, "data/train_md17/"),
                           name="aspirin CCSD",
                           train=True,
                           pre_transform=preprocess,
                           force_reload=True)

train_md17_dataset = modelling_md17_dataset[:int(len(modelling_md17_dataset)*0.85)]
valid_md17_dataset = modelling_md17_dataset[int(len(modelling_md17_dataset)*0.85):]

test_md17_dataset = datasets.MD17(root=Path(_dh[-1]) / os.path.join(additional_path, "data/test_md17/"),
                          name="aspirin CCSD",
                          train=False,
                          pre_transform=preprocess,
                          force_reload=True)

In [ ]:
train_md17_dataset[0]  # we have the edge attributes now

In [ ]:
train_md17_loader = loader.DataLoader(train_md17_dataset, batch_size=64, shuffle=True)
valid_md17_loader = loader.DataLoader(valid_md17_dataset, batch_size=64)
test_md17_loader = loader.DataLoader(test_md17_dataset, batch_size=64)

Let's first see how the GIN performes on the coordinates directly (not normalized!).

In [ ]:
class GCN_MD17(nn.Module):
    def __init__(
            self,
            embedding_in_features: int,
            embedding_hid_features: int,
            gnn_hid_features: int,
            gnn_num_layers: int,
            mlp_hid_layers: list[int],
            out_features: int,
            mlp_dropout: float
    ) -> None:
        super().__init__()

        self.embed = nn.Embedding(embedding_in_features, embedding_hid_features)
        total_gnn_hid_features = embedding_hid_features + 3  # 3 positional features

        gnn_blocks = [
            (geometric_nn.conv.GCNConv(total_gnn_hid_features, gnn_hid_features, add_self_loops=True), "x, edge_index -> x")
        ]

        for i in range(gnn_num_layers-1):
            gnn_blocks.append(
                (geometric_nn.conv.GCNConv(gnn_hid_features, gnn_hid_features, add_self_loops=True), "x, edge_index -> x")
                                )
            gnn_blocks.append(nn.LeakyReLU(inplace=True))

        gnn_blocks.append(
            (geometric_nn.global_mean_pool, "x, batch -> x")
                         )
        
        mlp_hid_layers += [out_features]
        
        gnn_blocks.append(
            (nn.Linear(gnn_hid_features, mlp_hid_layers[0]))
                          )

        mlp_blocks = []
        for (in_feat, out_feat) in zip(mlp_hid_layers[:-1], mlp_hid_layers[1:]):
            mlp_blocks.append(nn.BatchNorm1d(in_feat))
            mlp_blocks.append(nn.ReLU())
            mlp_blocks.append(nn.Dropout(mlp_dropout))
            mlp_blocks.append(nn.Linear(in_feat, out_feat))

        self.gnn_model = geometric_nn.Sequential("x, edge_index, batch", gnn_blocks)
        self.mlp_model = nn.Sequential(*mlp_blocks)

    def forward(self, batch: data.Batch) -> Tensor:
        x = self.embed(batch.z)
        x = torch.cat((x, batch.pos), dim=1)
        x = self.gnn_model(x, batch.edge_index, batch.batch)
        return self.mlp_model(x)

In [ ]:
simple_md17_model = GCN_MD17(
    embedding_in_features=10,
    embedding_hid_features=100,
    gnn_hid_features= 100,
    gnn_num_layers=3,
    mlp_hid_layers=[50, 20, 10],
    out_features=1,
    mlp_dropout=0.2
).to(DEVICE)

In [ ]:
trained_simple_md17_model, train_loss, valid_loss = train_model(
    simple_md17_model, 
    nn.MSELoss, 
    [],
    optim.Adam, 
    [0.01], 
    train_md17_loader, 
    valid_md17_loader,
    ["mae", "rmse"],
    "energy"
)

In [ ]:
epochs = [i+1 for i in range(len(train_loss))]

plt.plot(epochs, train_loss, color="blue", label="train loss")
plt.plot(epochs, valid_loss, color="red", label="valid loss")

plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("GCN model, MD17")

plt.legend()
plt.show()

In [ ]:
metrics_simple = test_model(trained_simple_md17_model, test_md17_loader, ["mae", "rmse"], "energy")
metrics_simple

Let's look at the invariant model.

In [ ]:
class InvariantMPLayer(nn.Module):
    def __init__(self,
                 node_in_feat: int,
                 edge_attr_feat: int,
                 out_feat: int
                 ):
        super().__init__()
    
        self.out_feat = out_feat  # also needed to bypass the not implemented backprop of scatter_reduce_ for index and src of not the same shape
        self.fc = nn.Linear(node_in_feat+edge_attr_feat, out_feat)

    def forward(self, x, edge_index, edge_attr) -> Tensor:
        source, target = edge_index
        
        x = torch.cat((x[source], edge_attr), dim=1)
        x = self.fc(x)

        target = target.unsqueeze(1).repeat(1, self.out_feat)

        return torch.zeros(torch.max(target).item()+1, self.out_feat, device=x.device).scatter_reduce_(
            dim=0,
            index=target,
            src=x,
            reduce="sum"
        )

class InvariantGNN(nn.Module):
    def __init__(self,
                 node_in_feat: int,
                 edge_attr_feat: int,
                 output_feat: int,
                 
                 embed_feat: int,
                 hid_feat: int,
                 num_gnn_layers: int,
                 gnn_act_func: nn.Module,

                 mlp_layers: list[int],
                 mlp_act_func: nn.Module,
                 mlp_batchnorm: bool,
                 mlp_dropout: float
                 ):  # one could also add bias terms and batchnorm of the GNN output
        super().__init__()

        self.embed = nn.Embedding(node_in_feat, embed_feat)
        
        gnn_blocks = []

        for i in range(num_gnn_layers):
            if i == 0:
                gnn_blocks.append(
                    (InvariantMPLayer(embed_feat, edge_attr_feat, hid_feat), "x, edge_index, edge_attr -> x")
                )
            else:
                gnn_blocks.append(
                    (InvariantMPLayer(hid_feat, edge_attr_feat, hid_feat), "x, edge_index, edge_attr -> x")
                )
            gnn_blocks.append(
                (gnn_act_func(), "x -> x")
            )

        gnn_blocks.append(
            (geometric_nn.global_add_pool, "x, batch -> x")
        )
            
        self.gnn = geometric_nn.Sequential("x, edge_index, edge_attr, batch", gnn_blocks)

        mlp_layers += [output_feat]

        mlp_blocks = [
            nn.Linear(hid_feat, mlp_layers[0])
        ]

        if len(mlp_layers) > 1:
            for (in_f, out_f) in zip(mlp_layers[:-1], mlp_layers[1:]):
                if mlp_batchnorm:
                    mlp_blocks.append(
                        nn.BatchNorm1d(in_f)
                    )
                mlp_blocks.extend(
                    [
                        mlp_act_func(),
                        nn.Dropout(mlp_dropout),
                        nn.Linear(in_f, out_f)
                    ]
                )

        self.mlp = nn.Sequential(*mlp_blocks)


    def forward(self, item_data: data.Batch) -> Tensor:
        x = self.embed(item_data.z)
        gnn_output = self.gnn(x, item_data.edge_index, item_data.edge_attr, item_data.batch)
        return self.mlp(gnn_output)

In [ ]:
invariant_model = InvariantGNN(
    node_in_feat=10,
    edge_attr_feat=1,
    output_feat=1,
    embed_feat=100,
    hid_feat=300,
    num_gnn_layers=3,
    gnn_act_func=nn.ReLU,
    mlp_layers=[200, 100, 1],
    mlp_act_func=nn.ReLU,
    mlp_batchnorm=True,
    mlp_dropout=0.2
).to(DEVICE)

In [ ]:
trained_invariant_model, valid_loss, test_loss = train_model(
    invariant_model, 
    nn.MSELoss, 
    [],
    optim.Adam, 
    [0.01], 
    train_md17_loader, 
    valid_md17_loader,
    ["mae", "rmse"],
    "energy"
)

In [ ]:
epochs = [i+1 for i in range(len(train_loss))]

plt.plot(epochs, train_loss, color="blue", label="train loss")
plt.plot(epochs, valid_loss, color="red", label="valid loss")

plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Invariant model, MD17")

plt.legend()
plt.show()

In [ ]:
metrics_invariant = test_model(trained_invariant_model, test_md17_loader, ["mae", "rmse"], "energy")
metrics_invariant

And now let's showcase how to create a simple equivariant model using spherical harmonics.

In [ ]:
class EquivariantLayer(torch.nn.Module):
    def __init__(self, in_node, in_edge, out_node, num_basis):
        super().__init__()
        self.num_basis = num_basis

        # The TensorProduct is the core "math" engine of e3nn
        self.tp = o3.FullyConnectedTensorProduct(
            in_node, 
            in_edge, 
            out_node,
            shared_weights=False
        )

        self.fc = e3nn_nn.FullyConnectedNet(
            [num_basis, 16, self.tp.weight_numel],
            torch.relu
        )

    def forward(self, x, edge_index, edge_vec):
        # edge_vec should be (pos[i] - pos[j])
        # Convert relative vectors to Spherical Harmonics
        sh = o3.spherical_harmonics(
            l="1x0e + 1x1e + 1x2e",
            x=edge_vec, 
            normalize=True, 
            normalization='component'
        )

        edge_length_embedding = soft_one_hot_linspace(
            edge_vec.norm(dim=1),
            start=0.0,
            end=CUTOFF,
            number=self.num_basis,
            basis="smooth_finite",
            cutoff=True
        )
        edge_length_embedding = edge_length_embedding * (self.num_basis ** 0.5)
        
        # Aggregate neighbors
        source, target = edge_index
        # Tensor product of node features and edge geometric features
        weight = self.fc(edge_length_embedding)
        
        out = self.tp(x[source], sh, weight)
        
        num_neighbors = torch.bincount(source).reshape(-1, 1)

        # Scatter sum to get back to node dimensionality
        # Using a simple mean or sum reduction
        final_nodes = torch.zeros((x.size(0), out.size(1)), device=x.device)
        final_nodes = final_nodes.index_add_(0, target, out) / (num_neighbors ** 0.5)

        return final_nodes
    

class EquivariantNetwork(nn.Module):
    def __init__(self, in_embed_feat, out_embed_feat, force_irreps, in_node_irreps, in_edge_irreps, out_node_ireps, num_basis,
                 mlp_layers, out_dim, mlp_dropout):
        super().__init__()
        self.force_irreps = force_irreps

        self.emb = nn.Embedding(in_embed_feat, out_embed_feat)

        self.gate = e3nn_nn.Gate(
            "16x0e + 16x0o",
            [torch.relu, torch.abs],  # scalar
            "8x0e + 8x0o + 8x0e + 8x0o",
            [torch.relu, torch.tanh, torch.relu, torch.tanh],  # gates scalars
            "16x1o + 16x1e"  # gated tensors
        )
        self.conv = EquivariantLayer(in_node_irreps, in_edge_irreps, self.gate.irreps_in, num_basis)
        self.final = EquivariantLayer(self.gate.irreps_out, in_edge_irreps, out_node_ireps, num_basis)

        mlp_blocks = []
        mlp_layers += [out_dim]

        mlp_blocks.append(nn.Linear(out_node_ireps.dim, mlp_layers[0]))

        for (in_feat, out_feat) in zip(mlp_layers[:-1], mlp_layers[1:]):
            mlp_blocks.extend(
                [
                nn.BatchNorm1d(in_feat),
                nn.ReLU(),
                nn.Dropout(mlp_dropout),
                nn.Linear(in_feat, out_feat)
                ]
            )

        self.mlp = nn.Sequential(*mlp_blocks)

    def forward(self, item_data: data.Batch) -> Tensor:
        z, pos, edge_index, force, batch = item_data.z, item_data.pos, item_data.edge_index, item_data.force, item_data.batch
        
        force_sh = o3.spherical_harmonics(self.force_irreps, force, normalize=True, normalization="component")

        x = torch.concat((self.emb(z),
                  force.norm(dim=1, keepdim=True),
                  force_sh),
                dim=1)

        source, target = edge_index
        edge_vec = pos[target] - pos[source]

        x = self.conv(x, edge_index, edge_vec)
        x = self.gate(x)
        x = self.final(x, edge_index, edge_vec)

        batch_repeats = batch.unsqueeze(1).repeat(1, x.shape[1])  # because backprop of scatter_reduce_ for index and src of not the same shape is not implemented

        x = torch.zeros((len(batch), x.shape[1]), device=x.device).scatter_reduce_(
            dim=0, 
            index=batch_repeats, 
            src=x, 
            reduce="sum"
        )

        x = geometric_nn.global_add_pool(x, batch)

        return self.mlp(x)

In [ ]:
equivariant_model = EquivariantNetwork(
    10,
    100,
    o3.Irreps.spherical_harmonics(2),  # forces are vectors, not pseudovectors
    o3.Irreps("102x0e + 1x1o + 1x2e"),
    o3.Irreps.spherical_harmonics(2),  # edges are vectors, not pseudovectors
    o3.Irreps("16x0e"),
    10,
    [8, 4],
    1,
    0.2
).to(DEVICE)

In [ ]:
# it takes a bit longer to train
trained_equivariant_model, train_loss, valid_loss = train_model(
    equivariant_model, 
    nn.MSELoss, 
    [], 
    optim.Adam, 
    [0.01], 
    train_md17_loader, 
    valid_md17_loader,
    ["mae", "rmse"],
    "energy"
)

In [ ]:
epochs = [i+1 for i in range(len(train_loss))]

plt.plot(epochs, train_loss, color="blue", label="train loss")
plt.plot(epochs, valid_loss, color="red", label="valid loss")

plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Equivariant model, MD17")

plt.legend()
plt.show()

In [ ]:
metrics_equivariant = test_model(trained_equivariant_model, test_md17_loader, ["mae", "rmse"], "energy")
metrics_equivariant

In [ ]:
metrics_equivariant.stored["mae"]

In [ ]:
# while developing this notebook, I had other models training in the background and I didn't want to risk crashing those sessions

del equivariant_model
# del trained_equivariant_model

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 8))

fig.suptitle("MD17 Energy Prediction Test Metrics")

bar_colors = ["tab:blue", "tab:red", "tab:green"]

ax[0].bar([
    "Naïve GCN", "Invariant", "Equivariant"
],
[
    metrics_simple.stored["mae"][0],
    metrics_invariant.stored["mae"][0],
    metrics_equivariant.stored["mae"][0]
], color=bar_colors)
ax[0].set_title("MAE")
ax[0].set_ylabel("Value")

ax[1].bar([
    "Naïve GCN", "Invariant", "Equivariant"
],
[
    metrics_simple.stored["rmse"][0],
    metrics_invariant.stored["rmse"][0],
    metrics_equivariant.stored["rmse"][0]
], color=bar_colors)
ax[1].set_title("RMSE")

fig.tight_layout()
plt.show()

As one can see, not only is the error for the naïve model higher (as it couldn't incorporate as much information). To be honest, this is not a very fair comparison, as the atom positions were given to the naïve model as their raw, centered values - a better practice would have been to scale them beforehand.

However, even if we did that, we can show that its outputs are not invariant to rotation anyway.

In [ ]:
batch = next(iter(train_md17_loader))
batch.to(DEVICE)

rotation_matrix = o3.rand_matrix().to(DEVICE)  # some random rotation

naive_output_before_rotation = trained_simple_md17_model(batch)
invariant_output_before_rotation = trained_invariant_model(batch)
equivariant_output_before_rotation = trained_equivariant_model(batch)

batch.pos = batch.pos @ rotation_matrix.T
batch.force = batch.force @ rotation_matrix.T

naive_output_after_rotation = trained_simple_md17_model(batch)
invariant_output_after_rotation = trained_invariant_model(batch)
equivariant_output_after_rotation = trained_equivariant_model(batch)

# does rotation keep the predictions the same?
print("Naïve GCN:", torch.all((naive_output_after_rotation - naive_output_before_rotation) < 1e-4).item())
print("Invariant model:", torch.all((invariant_output_after_rotation - invariant_output_before_rotation) < 1e-4).item())
print("Equivariant model:", torch.all((equivariant_output_after_rotation - equivariant_output_before_rotation) < 1e-4).item())

In [ ]:
del trained_equivariant_model

if torch.cuda.is_available():
    torch.cuda.empty_cache()

And what if we wanted to predict the forces (vectors)? In that case, we can only use an equivariant model.

In [ ]:
class EquivariantForceNetwork(nn.Module):
    def __init__(self, in_embed_feat, out_embed_feat, in_node_irreps, in_edge_irreps, out_node_ireps, num_basis):
        super().__init__()

        self.emb = nn.Embedding(in_embed_feat, out_embed_feat)

        self.gate = e3nn_nn.Gate(
            "16x0e + 16x0o",
            [torch.relu, torch.abs],  # scalar
            "8x0e + 8x0o + 8x0e + 8x0o",
            [torch.relu, torch.tanh, torch.relu, torch.tanh],  # gates scalars
            "16x1o + 16x1e"  # gated tensors
        )
        self.conv = EquivariantLayer(in_node_irreps, in_edge_irreps, self.gate.irreps_in, num_basis)
        self.final = EquivariantLayer(self.gate.irreps_out, in_edge_irreps, out_node_ireps, num_basis)

    def forward(self, item_data: data.Batch) -> Tensor:
        z, pos, edge_index, energy, ptr = item_data.z, item_data.pos, item_data.edge_index, item_data.energy, item_data.ptr
        
        x = torch.concat(
            (
                self.emb(z),
                torch.repeat_interleave(energy, ptr[1]-ptr[0], 0).view(-1, 1)
            ), 
            dim=1
        )

        source, target = edge_index
        edge_vec = pos[target] - pos[source]

        x = self.conv(x, edge_index, edge_vec)
        x = self.gate(x)
        x = self.final(x, edge_index, edge_vec)

        return x

In [ ]:
equivariant_force_model = EquivariantForceNetwork(
    10,
    100,
    o3.Irreps("101x0e"),
    o3.Irreps.spherical_harmonics(2),  # edges are vectors, not pseudovectors
    o3.Irreps("1x1o"),
    10,
).to(DEVICE)

In [ ]:
# it takes a bit longer to train
trained_equivariant_force_model, train_loss, valid_loss = train_model(
    equivariant_force_model, 
    nn.CosineEmbeddingLoss,
    [],
    optim.Adam, 
    [0.01], 
    train_md17_loader, 
    valid_md17_loader,
    ["cosine_similarity"],
    "force",
    True
)

In [ ]:
epochs = [i+1 for i in range(len(train_loss))]

plt.plot(epochs, train_loss, color="blue", label="train loss")
plt.plot(epochs, valid_loss, color="red", label="valid loss")

plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Equivariant force model, MD17")

plt.legend()
plt.show()

In [ ]:
metrics_equivariant_force = test_model(trained_equivariant_force_model, test_md17_loader, ["cosine_similarity"], "force")
metrics_equivariant_force

In [ ]:
# while developing this notebook, I had other models training in the background and I didn't want to risk crashing those sessions

del equivariant_force_model
# del trained_equivariant_force_model

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Again, we can show that the predictions are equivariant to rotation.

In [ ]:
batch = next(iter(train_md17_loader))
batch.to(DEVICE)

rotation_matrix = o3.rand_matrix().to(DEVICE)  # some random rotation

equivariant_output_before_rotation = trained_equivariant_force_model(batch)
equivariant_output_before_rotation = equivariant_output_before_rotation @ rotation_matrix.T  # rotate the output

# rotate the inputs
batch.pos = batch.pos @ rotation_matrix.T
batch.force = batch.force @ rotation_matrix.T

equivariant_output_after_rotation = trained_equivariant_force_model(batch)

# are the results equivalent?
print("Equivariant model - equivalent results:", torch.all((equivariant_output_after_rotation - equivariant_output_before_rotation) < 1e-4).item())

In [ ]:
del trained_equivariant_force_model

if torch.cuda.is_available():
    torch.cuda.empty_cache()